Сбор данных через API

Наша предметная область - это видеоигры, поэтому как источник для апи мы выбрали RAWG IO

Краткое описание источника

RAWG дает большой доступ к открытой базе данных игр и предоставляет структурированные данные по более 500000 играм 
на более 50 различных платформ

Источник дает нам возможность брать метаданные такие как: название, дата выхода, жанры, разработчики и публикаторы
Так же платформа предоставляет нам информацию по таким метрикам как: агрегированные оценки Metacritic и внутренняя система оценок RAWG
Магазины: ссылки на площадки цифровой дистрибуции, где можно приобрести игру.

Технические детали и условия использования
Тип запросов: API работает по протоколу HTTP REST, возвращая данные в формате JSON.
Аутентификация: для доступа ко всем эндпоинтам требуется персональный API ключ

И самое важное и полезное - Документация: подробное описание всех методов и параметров доступно на официальной странице

In [ ]:
import time
import logging
import pandas as pd
import requests
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")
url = "https://api.rawg.io/api"

logging.basicConfig(
    filename="rawg_api.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

s = requests.Session()
s.headers.update(
    {
        "Accept": "application/json",
    }
)


def get_json(point, p=None):
    if p is None:
        p = {}
    else:
        p = dict(p)

    p["key"] = api_key

    try:
        r = s.get(url + "/" + point, params=p)
    except Exception as e:
        logging.error("error %s %s", point, e)
        return 

    logging.info(
        "GET %s status=%s page=%s stores=%s",
        point,
        r.status_code,
        p.get("p"),
        p.get("stores"),
    )

    if r.status_code >= 400:
        return None

    return r.json()

def get_info(e, ps):
    res = []

    for p in range(1, ps + 1):
        d = get_json(e, {"page": p, "page_size": 40})
        if d is None:
            break

        temp = d.get("results") or []
        if not temp:
            break

        res.extend(temp)

        if not d.get("next"):
            break

        time.sleep(0.1)

    return pd.json_normalize(res)
    

def do_joining(arr, df=None):
    if arr == None or arr == []:
        return None

    res = []
    i = 0

    while i < len(arr):
        tmp = arr[i]

        if tmp != None:
            x = tmp

            plt = tmp.get("platform")
            st = tmp.get("store")

            if plt != None:
                x = plt
            else:
                if st != None:
                    x = st

            n = None

            if df != None:
                if x.get("id") != None:
                    if "id" in df.columns:
                        a = df[df["id"] == x.get("id")]

                        if len(a) > 0:
                            rowx = a.iloc[0]
                            val = rowx.get("name")

                            if val != None:
                                n = str(val).strip()

            if n == None:
                tmp_name = x.get("name")
                if tmp_name != None:
                    n = str(tmp_name).strip()

            if n != None and n != "":
                res.append(n)

        i = i + 1

    if res == []:
        return None

    ans = []
    temp = 0

    while temp < len(res):
        v = res[temp]
        if v not in ans:
            ans.append(v)
        temp = temp + 1

    return ", ".join(ans)


def has_steam(game):
    stores = game.get("stores")
    if stores == None:
        stores = []

    i = 0
    while i < len(stores):
        item = stores[i]

        if item != None:
            name = item.get("name")
            if name == "Steam":
                return True

            store = item.get("store")
            if store != None:
                store_name = store.get("name")
                if store_name == "Steam":
                    return True

                store_id = store.get("id")
                if store_id == 1:
                    return True

        i = i + 1

    return False


genres_df = get_info("genres", 2)
platforms_df = get_info("platforms", 2)
developers_df = get_info("developers", 3)
publishers_df = get_info("publishers", 3)
stores_df = get_info("stores", 2)

today = str(pd.Timestamp.now().date())
yesterday_ts = pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
yesterday = str(yesterday_ts.date())

upcoming_rows = []
p = 1

while True:
    data = get_json(
        "games",
        {
            "dates": today + ",2035-12-31",
            "page": p,
            "page_size": 40,
        },
    )

    if data is None:
        break

    b = data.get("results") or []
    if not b:
        break

    for game in b:
        game_id = game.get("id")
        if not game_id:
            continue

        upcoming_rows.append(
            {
                "id": game.get("id"),
                "name": str(game.get("name")).strip() if game.get("name") is not None else None,
                "released": game.get("released"),
                "dataset_label": "upcoming",
                "year": today + "..2035-12-31",
                "slug": game.get("slug"),
                "rating": game.get("rating"),
                "ratings_count": game.get("ratings_count"),
                "reviews_count": game.get("reviews_count"),
                "playtime": game.get("playtime"),
                "added": game.get("added"),
                "suggestions_count": game.get("suggestions_count"),
                "genres": do_joining(game.get("genres"), genres_df),
                "tags": do_joining(game.get("tags"), None),
                "platforms": do_joining(game.get("platforms"), platforms_df),
                "stores": do_joining(game.get("stores"), stores_df),
                "developers": do_joining(game.get("developers"), developers_df),
                "publishers": do_joining(game.get("publishers"), publishers_df),
            }
        )

    if not data.get("next"):
        break

    p += 1
    time.sleep(0.35)

upcoming_df = pd.DataFrame(upcoming_rows)
released_rows = []

for y in range(2026, 1999, -1):
    if len(released_rows) >= 10000 - len(upcoming_df):
        break

    start = str(y) + "-01-01"
    end = str(y) + "-12-31"
    if y == 2026:
        end = yesterday

    for p in range(1, 13):
        if len(released_rows) >= 10000 - len(upcoming_df):
            break

        data = get_json(
            "games",
            {
                "dates": start + "," + end,
                "page": p,
                "page_size": 40,
                "stores": 1,
            },
        )

        if data is None:
            break

        b = data.get("results") or []
        if not b:
            break

        for game in b:
            if len(released_rows) >= 10000 - len(upcoming_df):
                break

            game_id = game.get("id")
            if not game_id:
                continue
            if not game.get("name"):
                continue
            if not game.get("released"):
                continue
            if not game.get("genres") or not game.get("platforms"):
                continue
            if game.get("rating") is None:
                continue
            if (game.get("ratings_count") or 0) <= 0:
                continue
            if (game.get("reviews_count") or 0) <= 0:
                continue
            if not has_steam(game):
                continue

            detail = get_json("games/" + str(game_id))
            if detail is None:
                continue

            if not detail.get("name"):
                continue
            if not detail.get("released"):
                continue
            if not detail.get("genres") or not detail.get("platforms"):
                continue
            if detail.get("rating") is None:
                continue
            if (detail.get("ratings_count") or 0) <= 0:
                continue
            if (detail.get("reviews_count") or 0) <= 0:
                continue
            if not has_steam(detail):
                continue

            released_rows.append(
                {
                    "id": detail.get("id"),
                    "name": str(detail.get("name")).strip() if detail.get("name") is not None else None,
                    "released": detail.get("released"),
                    "dataset_label": "released",
                    "year": str(y),
                    "slug": detail.get("slug"),
                    "rating": detail.get("rating"),
                    "ratings_count": detail.get("ratings_count"),
                    "reviews_count": detail.get("reviews_count"),
                    "playtime": detail.get("playtime"),
                    "added": detail.get("added"),
                    "suggestions_count": detail.get("suggestions_count"),
                    "genres": do_joining(detail.get("genres"), genres_df),
                    "tags": do_joining(detail.get("tags")),
                    "platforms": do_joining(detail.get("platforms"), platforms_df),
                    "stores": do_joining(detail.get("stores"), stores_df),
                    "developers": do_joining(detail.get("developers"), developers_df),
                    "publishers": do_joining(detail.get("publishers"), publishers_df),
                }
            )

        if not data.get("next"):
            break

        time.sleep(0.35)

released_df = pd.DataFrame(released_rows)

upcoming_df.to_csv("rawg_upcoming_games_api.csv", index=False, encoding="utf-8-sig")
released_df.to_csv("rawg_released_games_api.csv", index=False, encoding="utf-8-sig")


Сбор данных через скрепинг

Краткое описание источника

Steam является крупнейшей платформой цифровой дистрибуции игр, где представлена информация о тысячах проектов
На страницах игр можно получить дополнительные данные, которых нет в RAWG

Источник дает нам возможность брать такие данные как: процент положительных отзывов, количество отзывов, текстовая оценка, цена игры и скидки
Также можно получить информацию о поддерживаемых платформах и уникальный идентификатор игры (steam_app_id), что важно для объединения датасетов

Технические детали и условия использования

Тип запросов: данные получаются через http запросы и парсинг html страниц
Используемые инструменты: requests и BeautifulSoup для извлечения данных из html

Дополнительно использовался Selenium, так как часть данных на сайте подгружается динамически и может быть недоступна при обычных запросах

Данные со Steam позволили дополнить API датасет реальными пользовательскими метриками и подготовить итоговый объединенный датасет для анализа


In [ ]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver

df = pd.read_csv("rawg_released_games_api.csv")

driver = webdriver.Chrome()

data = []

for i in range(len(df)):
    name = df.loc[i, "name"]

    steam_review_percent = ""
    steam_review_count = ""
    steam_review_label = ""
    steam_price_text = ""
    steam_discount_pct = ""
    steam_platform_flags = ""
    steam_app_id = ""

    url = "https://store.steampowered.com/search/?term=" + str(name).replace(" ",
                                                                             "+") + "&supportedlang=english&ignore_preferences=1"
    driver.get(url)
    time.sleep(0.5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    karta = soup.select_one("a.search_result_row")

    if karta is not None:
        steam_app_id = karta.get("data-ds-appid", "")

        r_t = karta.select_one("span.search_review_summary")
        if r_t is not None:
            t = r_t.get("data-tooltip-html", "")
            if t != "":
                txt = BeautifulSoup(t, "html.parser").get_text(" ", strip=True)

                if "%" in txt:
                    a = txt.split("%")[0]
                    num = ""
                    j = len(a) - 1
                    while j >= 0:
                        if a[j].isdigit():
                            num = a[j] + num
                        elif num != "":
                            break
                        j -= 1
                    steam_review_percent = num

                if "of the" in txt and "user reviews" in txt:
                    a = txt.split("of the", 1)[1]
                    a = a.split("user reviews", 1)[0]
                    a = a.replace(",", "").strip()
                    num = ""
                    for e in a:
                        if e.isdigit():
                            num += e
                    steam_review_count = num

                if "%" in txt:
                    a = txt.split("%")[0].strip()
                    while len(a) > 0 and a[-1].isdigit():
                        a = a[:-1].strip()
                    steam_review_label = a

        p_tag = karta.select_one("div.discount_final_price")
        if p_tag is None:
            p_tag = karta.select_one("div.search_price")
        if p_tag is None:
            p_tag = karta.select_one("div.search_price_discount_combined")
        if p_tag is not None:
            steam_price_text = p_tag.get_text(" ")

        d_tag = karta.select_one("div.discount_pct")
        if d_tag is not None:
            steam_discount_pct = d_tag.get_text(" ").replace("-", "").replace("%", "").strip()

        f = []
        for sp in karta.select("span.platform_img"):
            for c in sp.get("class", []):
                if c != "platform_img" and c not in f:
                    f.append(c)

        steam_platform_flags = ", ".join(f)

    data.append({
        "name": name,
        "steam_review_percent": steam_review_percent,
        "steam_review_count": steam_review_count,
        "steam_review_label": steam_review_label,
        "steam_price_text": steam_price_text,
        "steam_discount_pct": steam_discount_pct,
        "steam_platform_flags": steam_platform_flags,
        "steam_app_id": steam_app_id
    })

    time.sleep(0.5)

driver.quit()

result = pd.DataFrame(data)
result.to_csv("final_scraping_dataset_10000.csv", index=False, encoding="utf-8-sig")
